# Shark Attack Modeling (Refactored)

This notebook uses the shared `src/` modules for preprocessing and model evaluation.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.modeling import (
    cross_validate_pr_auc,
    evaluate_classifier,
    get_baseline_models,
    make_train_test_split,
    summarize_results,
)
from src.preprocessing import build_model_dataset, prepare_attacks_dataframe

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "attacks_cleaned.csv"

## Load the prepared dataframe

In [ ]:
df = prepare_attacks_dataframe(DATA_PATH)
df.shape

## Build model features

In [ ]:
X_encoded, y, model_df = build_model_dataset(df)

print("Model dataframe shape:", model_df.shape)
print("Encoded feature matrix shape:", X_encoded.shape)
y.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = make_train_test_split(X_encoded, y)
print(X_train.shape, X_test.shape)

## Baseline models

In [ ]:
models = get_baseline_models()
results = {}

for name, model in models.items():
    results[name] = evaluate_classifier(model, X_train, X_test, y_train, y_test)

results_table = summarize_results(results)
results_table

## Detailed look at logistic regression

In [ ]:
log_reg = results["Logistic Regression"]
pd.DataFrame(log_reg["test_report"]).T

## Cross-validated PR AUC

In [ ]:
cv_rows = []
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        mean_pr_auc, std_pr_auc = cross_validate_pr_auc(model, X_encoded, y)
        cv_rows.append(
            {"model": name, "cv_pr_auc_mean": mean_pr_auc, "cv_pr_auc_std": std_pr_auc}
        )

pd.DataFrame(cv_rows).sort_values("cv_pr_auc_mean", ascending=False)

## Optional gradient boosting models

In [ ]:
optional_rows = []

try:
    from xgboost import XGBClassifier

    xgb_model = XGBClassifier(
        random_state=42,
        eval_metric="logloss",
    )
    xgb_results = evaluate_classifier(
        xgb_model,
        X_train,
        X_test,
        y_train.map({"N": 0, "Y": 1}),
        y_test.map({"N": 0, "Y": 1}),
    )
    optional_rows.append(
        {
            "model": "XGBoost",
            "test_accuracy": xgb_results["test_report"]["accuracy"],
            "test_f1_fatal": xgb_results["test_report"].get("1", {}).get("f1-score"),
            "test_f1_nonfatal": xgb_results["test_report"].get("0", {}).get("f1-score"),
            "test_pr_auc": xgb_results.get("test_pr_auc"),
        }
    )
except Exception as exc:
    print(f"Skipping XGBoost: {exc}")

try:
    from lightgbm import LGBMClassifier

    lgbm_model = LGBMClassifier(class_weight="balanced", random_state=42, verbosity=-1)
    lgbm_results = evaluate_classifier(
        lgbm_model,
        X_train,
        X_test,
        y_train.map({"N": 0, "Y": 1}),
        y_test.map({"N": 0, "Y": 1}),
    )
    optional_rows.append(
        {
            "model": "LightGBM",
            "test_accuracy": lgbm_results["test_report"]["accuracy"],
            "test_f1_fatal": lgbm_results["test_report"].get("1", {}).get("f1-score"),
            "test_f1_nonfatal": lgbm_results["test_report"].get("0", {}).get("f1-score"),
            "test_pr_auc": lgbm_results.get("test_pr_auc"),
        }
    )
except Exception as exc:
    print(f"Skipping LightGBM: {exc}")

pd.DataFrame(optional_rows)